In [6]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()
model = ChatOpenAI(model='gpt-4o-mini')

In [2]:
class BlogState(TypedDict):
    
    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    
    # fetch_title
    title = state['title']
    
    # call llm to generate the outline
    prompt = f'Generate a detailed outline for a blog on the topic {title}'
    
    # invoke
    outline = model.invoke(prompt)
    
    # storing it to the state
    state['outline'] = outline
    
    return state

In [10]:
def create_blog(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f"Write a detailed blog on the title {title} using the following outline \n {outline}"
    
    blog = model.invoke(prompt).content
    
    state['content'] = blog
    
    return state

In [11]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()

In [12]:
initial_state = {'title':'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

final_state

{'title': 'Rise of AI in India',
 'outline': AIMessage(content="# Blog Outline: The Rise of AI in India\n\n## Introduction\n- Brief overview of AI and its significance globally.\n- Importance of AI for India: Economic growth, job creation, and technological advancement.\n- Purpose of the blog: To explore the current state, potential, and challenges of AI in India.\n\n## 1. Historical Context of AI in India\n   - Early developments in AI (1980s and 1990s).\n   - Key academic and research contributions.\n   - Government initiatives and policies that set the foundation for AI (e.g., Technology Development for Indian Languages).\n\n## 2. Current Landscape of AI in India\n   - Overview of the AI market in India.\n     - Market size and growth projections.\n     - Key sectors adopting AI: healthcare, finance, agriculture, education, etc.\n   - Major players in the AI ecosystem:\n     - Startups: Innovators and disruptors.\n     - Established companies: Tech giants and traditional industries.

In [14]:
print(final_state['content'])

# The Rise of AI in India

## Introduction

Artificial Intelligence (AI) has emerged as one of the pivotal forces shaping the global economy and technology landscape. As countries race to harness the power of AI, its significance for emerging economies, especially India, cannot be understated. For India, AI represents an opportunity for economic growth, job creation, and technological advancement, all essential for maintaining its pace in the global arena. This blog explores the current state, potential, and challenges of AI in India, tracing its journey from inception to the forefront of innovation.

## 1. Historical Context of AI in India

AI's roots in India can be traced back to the 1980s and 1990s. During this period, several academic institutions began interdisciplinary research, leading to foundational work in machine learning and knowledge-based systems. Prominent universities and institutes like the Indian Institute of Technology (IIT) and the Indian Statistical Institute (ISI